**VALUASI EKONOMI PM2.5 PER PROVINSI (2010-2025)**

Menggabungkan:
  1. Data GBD (DALY & Deaths attributable PM2.5), 2010-2023
  2. Pengisian celah 2024-2025 (rate 2023 dipertahankan x populasi)
  3. Validasi CDR nasional (justifikasi asumsi rate stabil)
  4. Valuasi DALY-based (metode utama)
  5. Valuasi VSL-based (metode sensitivitas), formula & parameter dari
     World Bank (2016), "The Cost of Air Pollution: Strengthening the
     Economic Case for Action", halaman 48-49
  6. Sanity-check terhadap benchmark nasional Indonesia dari laporan
     yang sama, halaman 95-96 (~5% PDB, tahun 2013)

Rujukan:
  - Apte, Marshall, Cohen, & Brauer (2015). "Addressing Global Mortality
    from Ambient PM2.5." Environmental Science & Technology.
  - Mathers & Loncar (2006). "Projections of Global Mortality and
    Burden of Disease from 2002 to 2030." PLoS Medicine.
  - World Bank (2016). "The Cost of Air Pollution: Strengthening the
    Economic Case for Action.

In [36]:
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
FOLDER_SUMBER = "/content/drive/MyDrive/DatabaseSkripsi/SumberData"
FOLDER_OUTPUT = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# =====================================================================
# BAGIAN 1: DAFTAR PROVINSI STANDAR & HARMONISASI
# =====================================================================
PROVINSI_STANDAR = [
    "ACEH", "BALI", "BANTEN", "BENGKULU", "DI YOGYAKARTA", "DKI JAKARTA",
    "GORONTALO", "JAMBI", "JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR",
    "KALIMANTAN BARAT", "KALIMANTAN SELATAN", "KALIMANTAN TENGAH",
    "KALIMANTAN TIMUR", "KALIMANTAN UTARA", "KEP. BANGKA BELITUNG",
    "KEP. RIAU", "LAMPUNG", "MALUKU", "MALUKU UTARA",
    "NUSA TENGGARA BARAT", "NUSA TENGGARA TIMUR", "PAPUA", "PAPUA BARAT",
    "RIAU", "SULAWESI BARAT", "SULAWESI SELATAN", "SULAWESI TENGAH",
    "SULAWESI TENGGARA", "SULAWESI UTARA", "SUMATERA BARAT",
    "SUMATERA SELATAN", "SUMATERA UTARA",
]

PEMETAAN_KHUSUS = {
    "DAERAH ISTIMEWA YOGYAKARTA": "DI YOGYAKARTA",
    "YOGYAKARTA": "DI YOGYAKARTA",
    "KEPULAUAN BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN BANGKA-BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN RIAU": "KEP. RIAU",
    "JAKARTA RAYA": "DKI JAKARTA",
    "JAKARTA": "DKI JAKARTA",
    "SUMATRA BARAT": "SUMATERA BARAT",
    "SUMATRA SELATAN": "SUMATERA SELATAN",
    "SUMATRA UTARA": "SUMATERA UTARA",
}


def normalisasi_dasar(nama):
    return re.sub(r"\s+", " ", str(nama).strip())


def harmonisasi_provinsi(nama):
    nama_upper = normalisasi_dasar(nama).upper()
    if nama_upper in PEMETAAN_KHUSUS:
        return PEMETAAN_KHUSUS[nama_upper]
    for standar in PROVINSI_STANDAR:
        if nama_upper == standar.upper():
            return standar
    return None  # termasuk "INDONESIA" (level nasional)

In [38]:
# =====================================================================
# BAGIAN 2: BACA & HARMONISASI DATA GBD (DALY & DEATHS)
# =====================================================================
gbd = pd.read_csv(f"{FOLDER_SUMBER}/CDR_pollution_ihme.csv")
gbd = gbd[gbd["metric_name"].isin(["Angka", "Laju"])]
gbd = gbd[gbd["measure_name"].isin(["Meninggal", "DALYs (Disability-Adjusted Life Years)"])]
gbd["PROVINSI"] = gbd["location_name"].apply(harmonisasi_provinsi)
gbd = gbd[gbd["PROVINSI"].notnull()]  # buang "Indonesia" (nasional)

gbd_pivot = gbd.pivot_table(
    index=["PROVINSI", "year"], columns=["measure_name", "metric_name"], values="val",
).reset_index()
gbd_pivot.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c for c in gbd_pivot.columns
]
gbd_pivot = gbd_pivot.rename(columns={
    "Meninggal_Angka": "kematian_jumlah",
    "Meninggal_Laju": "kematian_rate_per100rb",
    "DALYs (Disability-Adjusted Life Years)_Angka": "daly_jumlah",
    "DALYs (Disability-Adjusted Life Years)_Laju": "daly_rate_per100rb",
})
print(f"[GBD] {gbd_pivot['PROVINSI'].nunique()} provinsi, "
      f"tahun {gbd_pivot['year'].min()}-{gbd_pivot['year'].max()}, "
      f"{len(gbd_pivot)} baris (harapan 34x14=476)")

[GBD] 34 provinsi, tahun 2010-2023, 476 baris (harapan 34x14=476)


In [39]:
# =====================================================================
# BAGIAN 3: BACA & HARMONISASI POPULASI + PDRB PER KAPITA
# =====================================================================
penduduk = pd.read_csv(
    f"{FOLDER_SUMBER}/Jumlah_Penduduk.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
penduduk = penduduk.loc[:, ~penduduk.columns.str.contains("^Unnamed")]  # buang kolom kosong akibat ';' ekstra di akhir header
penduduk["PROVINSI"] = penduduk["Provinsi"].apply(harmonisasi_provinsi)
penduduk = penduduk.rename(columns={"Tahun": "year", "Penduduk": "populasi"})[
    ["PROVINSI", "year", "populasi"]
]
print(f"[Populasi] {penduduk['PROVINSI'].nunique()} provinsi (harapan 34)")

pdrb_kapita = pd.read_csv(
    f"{FOLDER_SUMBER}/PDRB_per_Kapita_ADHK.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
pdrb_kapita.columns = ["PROVINSI", "year", "pdrb_per_kapita_ribu_rp"]

[Populasi] 34 provinsi (harapan 34)


In [40]:
# =====================================================================
# BAGIAN 4: GABUNGKAN + ISI CELAH 2024-2025 (rate 2023 x populasi)
# =====================================================================
panel = penduduk.merge(pdrb_kapita, on=["PROVINSI", "year"], how="left")
panel = panel.merge(gbd_pivot, on=["PROVINSI", "year"], how="left")

rate_2023 = gbd_pivot[gbd_pivot["year"] == 2023][
    ["PROVINSI", "daly_rate_per100rb", "kematian_rate_per100rb"]
].rename(columns={
    "daly_rate_per100rb": "daly_rate_2023", "kematian_rate_per100rb": "kematian_rate_2023",
})

for tahun_target in [2024, 2025]:
    mask = panel["year"] == tahun_target
    pt = panel[mask].merge(rate_2023, on="PROVINSI", how="left")
    panel.loc[mask, "daly_jumlah"] = pt["daly_rate_2023"].values * pt["populasi"].values / 100000
    panel.loc[mask, "kematian_jumlah"] = pt["kematian_rate_2023"].values * pt["populasi"].values / 100000

panel["sumber_gbd"] = panel["year"].apply(
    lambda y: "GBD_langsung" if y <= 2023 else "estimasi_rate_2023_x_populasi"
)

kosong = panel["daly_jumlah"].isnull().sum()
print(f"[Celah] {kosong} baris masih kosong setelah pengisian (harapan: 0)")

[Celah] 0 baris masih kosong setelah pengisian (harapan: 0)


In [41]:
# =====================================================================
# BAGIAN 5: VALIDASI CDR (justifikasi asumsi rate stabil -- BUKAN input
# perhitungan langsung; lihat pembahasan sebelumnya soal kenapa CDR
# all-cause & nasional tidak dipakai sebagai pengganda langsung)
# =====================================================================
try:
    cdr = pd.read_csv(
        f"{FOLDER_SUMBER}/CDR_Indonesia.csv",
        sep=";", decimal=",", encoding="utf-8-sig"
    )
    cdr.columns = ["year", "cdr"]
    baseline_prapandemi = cdr[cdr["year"].between(2010, 2019)]["cdr"].mean()
    cdr_terkini = cdr[cdr["year"].between(2022, 2025)]["cdr"].mean()
    selisih_persen = abs(cdr_terkini - baseline_prapandemi) / baseline_prapandemi * 100

    print(f"\n[Validasi CDR] Rata-rata CDR 2010-2019 (pra-pandemi): {baseline_prapandemi:.3f}")
    print(f"[Validasi CDR] Rata-rata CDR 2022-2025 (pasca-pandemi): {cdr_terkini:.3f}")
    print(f"[Validasi CDR] Selisih: {selisih_persen:.2f}%")
    if selisih_persen < 5:
        print("[Validasi CDR] KESIMPULAN: kondisi mortalitas nasional 2022-2025 "
              "sudah kembali stabil mendekati level pra-pandemi -- mendukung "
              "asumsi rate attributable PM2.5 yang relatif stabil pada periode "
              "ekstrapolasi 2024-2025 (lihat Apte et al. 2015; Mathers & Loncar 2006).")
    else:
        print("[Validasi CDR] PERHATIAN: ada deviasi CDR >5% dari baseline -- "
              "pertimbangkan kembali asumsi rate konstan.")
except FileNotFoundError:
    print("\n[Validasi CDR] File CDR_Indonesia.xlsx tidak ditemukan -- validasi dilewati.")


[Validasi CDR] Rata-rata CDR 2010-2019 (pra-pandemi): 7.519
[Validasi CDR] Rata-rata CDR 2022-2025 (pasca-pandemi): 7.556
[Validasi CDR] Selisih: 0.50%
[Validasi CDR] KESIMPULAN: kondisi mortalitas nasional 2022-2025 sudah kembali stabil mendekati level pra-pandemi -- mendukung asumsi rate attributable PM2.5 yang relatif stabil pada periode ekstrapolasi 2024-2025 (lihat Apte et al. 2015; Mathers & Loncar 2006).


In [42]:
# =====================================================================
# BAGIAN 6: VALUASI DALY-BASED (METODE UTAMA)
# =====================================================================
FAKTOR_SATUAN_PDRB = 1000  # asumsi: ribu Rupiah -> Rupiah penuh (VERIFIKASI!)

panel["nilai_per_daly_1x"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB * 1
panel["nilai_per_daly_3x"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB * 3
panel["nilai_pm25_rp_daly_1x"] = panel["daly_jumlah"] * panel["nilai_per_daly_1x"]
panel["nilai_pm25_rp_daly_3x"] = panel["daly_jumlah"] * panel["nilai_per_daly_3x"]

In [43]:
# =====================================================================
# BAGIAN 7: VALUASI VSL-BASED (SENSITIVITAS)
#    Formula & parameter: World Bank (2016), hal. 48-49
#    VSL_c,n = VSL_OECD x (Y_c,n / Y_OECD) ^ e
# =====================================================================
VSL_OECD_USD = 3_830_000       # USD 3,83 juta (World Bank 2016, hal. 48)
Y_OECD_USD = 37_000            # USD 37.000 (World Bank 2016, hal. 48)
ELASTISITAS = 1.2              # negara berpendapatan rendah-menengah (hal. 49)
KURS_2010 = 8991               # konsisten dgn seri harga karbon sebelumnya

# PDRB per kapita (Rupiah penuh, konstan 2010) -> USD ekuivalen, memakai
# SATU kurs tetap (2010) untuk konsistensi dgn seluruh skripsi -- BUKAN
# penyesuaian PPP resmi (simplifikasi yang perlu dinyatakan eksplisit
# sebagai keterbatasan, tapi konsisten dgn pendekatan kurs-tetap yang
# sudah dipakai di perhitungan SCC).
panel["pdrb_per_kapita_rp"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB
panel["pdrb_per_kapita_usd"] = panel["pdrb_per_kapita_rp"] / KURS_2010

panel["vsl_provinsi_usd"] = VSL_OECD_USD * (
    panel["pdrb_per_kapita_usd"] / Y_OECD_USD
) ** ELASTISITAS
panel["vsl_provinsi_rp"] = panel["vsl_provinsi_usd"] * KURS_2010

panel["nilai_pm25_rp_vsl"] = panel["kematian_jumlah"] * panel["vsl_provinsi_rp"]

In [44]:
# =====================================================================
# BAGIAN 8: SANITY-CHECK TERHADAP BENCHMARK WORLD BANK (~5% PDB, 2013)
#    Memakai PDRB ADHK RESMI (bukan estimasi kasar per kapita x populasi)
# =====================================================================
pdrb_adhk = pd.read_csv(
    f"{FOLDER_SUMBER}/PDRB_ADHK_Rill.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
pdrb_adhk.columns = ["PROVINSI", "year", "pdrb_adhk_rp"]
pdrb_adhk["PROVINSI"] = pdrb_adhk["PROVINSI"].apply(harmonisasi_provinsi)

panel = panel.merge(pdrb_adhk, on=["PROVINSI", "year"], how="left")

cek_2013 = panel[panel["year"] == 2013]
if len(cek_2013) > 0 and cek_2013["pdrb_adhk_rp"].notnull().any():
    rasio_daly = cek_2013["nilai_pm25_rp_daly_1x"].sum() / cek_2013["pdrb_adhk_rp"].sum() * 100
    rasio_vsl = cek_2013["nilai_pm25_rp_vsl"].sum() / cek_2013["pdrb_adhk_rp"].sum() * 100
    print(f"\n[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (DALY-1x) / PDRB ADHK: {rasio_daly:.2f}%")
    print(f"[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (VSL)     / PDRB ADHK: {rasio_vsl:.2f}%")
    print(f"[Sanity-check 2013] Benchmark World Bank (2016, hal. 95-96): 5,15% PDB")
    print("[Sanity-check] Kalau rasio Anda jauh berbeda (misal >20% atau <0,5%), "
          "cek ulang satuan PDRB per kapita atau FAKTOR_SATUAN_PDRB.")
else:
    print("\n[Sanity-check] Tidak bisa dihitung -- cek kecocokan PROVINSI/year dgn PDRB_ADHK_Rill.csv.")


[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (DALY-1x) / PDRB ADHK: 2.65%
[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (VSL)     / PDRB ADHK: 4.96%
[Sanity-check 2013] Benchmark World Bank (2016, hal. 95-96): 5,15% PDB
[Sanity-check] Kalau rasio Anda jauh berbeda (misal >20% atau <0,5%), cek ulang satuan PDRB per kapita atau FAKTOR_SATUAN_PDRB.


In [45]:
# =====================================================================
# BAGIAN 9: VERIFIKASI & SIMPAN
# =====================================================================
print(f"\n{'='*70}\nRINGKASAN AKHIR\n{'='*70}")
print(f"Jumlah baris   : {len(panel)} (harapan 34x16=544)")
print(f"Jumlah provinsi: {panel['PROVINSI'].nunique()}")

print("\nContoh hasil (Aceh):")
kolom_tampil = ["year", "daly_jumlah", "kematian_jumlah", "nilai_pm25_rp_daly_1x",
                 "nilai_pm25_rp_vsl", "sumber_gbd"]
print(panel[panel["PROVINSI"] == "ACEH"][kolom_tampil].to_string(
    index=False, float_format=lambda x: f"{x:,.0f}"))

output_path = f"{FOLDER_OUTPUT}/panel_nilai_pm25_lengkap_2010_2025.csv"
panel.to_csv(output_path, index=False)
print(f"\nDisimpan ke: {output_path}")
print(
    "\nCATATAN METODOLOGI:\n"
    "- nilai_pm25_rp_daly_1x = metode UTAMA (1x PDRB per kapita per DALY).\n"
    "- nilai_pm25_rp_daly_3x = sensitivitas atas (3x PDRB per kapita).\n"
    "- nilai_pm25_rp_vsl     = sensitivitas via VSL (formula & parameter\n"
    "  World Bank 2016, hal. 48-49; PDRB per kapita dikonversi USD dgn\n"
    "  kurs tetap 2010, BUKAN penyesuaian PPP resmi -- keterbatasan yg\n"
    "  perlu dinyatakan eksplisit).\n"
    "- sumber_gbd membedakan data GBD asli (2010-2023) dari estimasi\n"
    "  perluasan (2024-2025, rate 2023 x populasi berjalan).\n"
    "- Validasi CDR mendukung asumsi rate stabil untuk perluasan tsb."
)


RINGKASAN AKHIR
Jumlah baris   : 544 (harapan 34x16=544)
Jumlah provinsi: 34

Contoh hasil (Aceh):
 year  daly_jumlah  kematian_jumlah  nilai_pm25_rp_daly_1x  nilai_pm25_rp_vsl                    sumber_gbd
 2010      117,615            3,116      2,640,480,455,531  4,222,875,395,560                  GBD_langsung
 2011      114,664            3,070      2,603,430,440,199  4,217,759,400,859                  GBD_langsung
 2012      115,284            3,117      2,662,964,238,044  4,371,621,425,208                  GBD_langsung
 2013      114,147            3,117      2,651,477,874,659  4,401,209,736,615                  GBD_langsung
 2014      112,956            3,110      2,612,571,284,778  4,368,856,661,640                  GBD_langsung
 2015      112,962            3,139      2,544,392,261,331  4,271,884,399,204                  GBD_langsung
 2016      110,229            3,080      2,517,113,549,020  4,260,185,841,085                  GBD_langsung
 2017      106,164            2,979 